# Set Ups

In [4]:
import os
# import uuid
import nest_asyncio
import asyncio
from fastapi import UploadFile
from llama_index.core import VectorStoreIndex, StorageContext, Settings, PromptTemplate
from llama_index.core.node_parser import MarkdownNodeParser
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
from llama_index.core.extractors import TitleExtractor
from llama_index.core.text_splitter import SentenceSplitter
from llama_index.readers.docling import DoclingReader
from llama_index.core.schema import Document
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
from llama_index.core.response_synthesizers import ResponseMode
from llama_index.core import get_response_synthesizer

import chromadb
from llama_index.core.retrievers import BaseRetriever
from typing import List, Dict, Any
import xml.etree.ElementTree as ET

from dotenv import load_dotenv
load_dotenv()

from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb
from llama_index.llms.ollama import Ollama
import torch
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from pathlib import Path
nest_asyncio.apply()
from llama_index.embeddings.gemini import GeminiEmbedding
from llama_index.llms.gemini import Gemini

os.getcwd()

'/Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python/_pipeline'

In [5]:
### PDD
# NOTE: Change things here
# MUST START WITH REGISTRY CODE & PROJECT CODE
p_code = '5458'
file_path_pdd = f'data/pdd/VCS_{p_code}.pdf'
print(file_path_pdd)

data/pdd/VCS_5458.pdf


In [6]:
if torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
    
def llm_setting(name = "Gemini"):
    if name == "Gemini": 
        from llama_index.llms.gemini import Gemini
        api_key = os.getenv("GEMINI_API_KEY")
        # define embedding model
        # embed_model = GeminiEmbedding(
        #     model_name="models/embedding-004", 
        #     api_key=api_key,
        #     embed_batch_size=10
        # )
        embed_model = GoogleGenAIEmbedding(
            model_name="text-embedding-004",
            api_key=api_key,
            embed_batch_size=100)
        # embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", 
        #                                    device=device,
        #                                    embed_batch_size=10)
        llm = Gemini(model_name="models/gemini-2.0-flash", 
                     temperature=0.1, 
                     max_tokens=100000, 
                     api_key=api_key)
    
    # if default -> use local model with HF embedding
    else:
        # api_key = os.getenv("HF_API_KEY")
        # define embedding model
        from llama_index.llms.ollama import Ollama
        from llama_index.embeddings.huggingface import HuggingFaceEmbedding
        embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5", 
                                           device=device,
                                           embed_batch_size=10)
        
        llm = Ollama(model="deepseek-r1:7b", 
                     request_timeout=120.0)
    return llm, embed_model

## Global setting

In [7]:
# Set up model
llm, embed_model = llm_setting()

# Set default LLM and embedding model 
Settings.llm = llm
Settings.embed_model = embed_model
# maybe need to define sentencesplitter chunk size
# Settings.text_splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=20)

chroma_client = chromadb.PersistentClient(path="./chroma_db_gemini")
reader = DoclingReader()
node_parser = MarkdownNodeParser()

API_TAV = os.getenv("TAVILY_API_KEY")

/var/folders/j2/yjk_0cz112g3l8vv2_013tmr0000gn/T/ipykernel_16800/4275260026.py:23: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/)
  llm = Gemini(model_name="models/gemini-2.0-flash",


In [8]:
collections = chroma_client.list_collections() 
collections

['policy_vcm', 'testing', 'regional_policy_IND', 'pdd_new']

# a1. PDF to LLamaIndex in ChromaDB - PDD (no Policy, Regional Policy)

### 1.0 Helper Functions

In [9]:
### Common Functions

# get all file paths from folder
def get_file_paths(folder_path):
    """Get all file paths in a folder recursively, excluding .DS_Store files."""
    file_paths = []
    for root, dirs, files in os.walk(folder_path):
        for file in files:
            if file != '.DS_Store':
                file_paths.append(os.path.join(root, file))
    return file_paths

import re
def rename_file_with_prefix(file_path, prefix='forestPolicy_IND'):
    """
    Renames a file by stripping whitespace and special characters from its name
    and adding the prefix 'forestPolicy_IND'.
    
    Args:
        file_path (str): The full path to the file (e.g., '/path/to/My File@#$.pdf')
        prefix (str): prefix of the document
    
    Returns:
        str: The new file path after renaming, or None if the file doesn't exist or an error occurs.
    
    Example:
        Input: '/path/to/My File@#$.pdf'
        Output: '/path/to/forestPolicy_IND_MyFile.pdf'
    """
    try:
        # Check if the file exists
        if not os.path.isfile(file_path):
            print(f"Error: File '{file_path}' does not exist.")
            return None
        
        # Get the directory and file name
        directory = os.path.dirname(file_path)
        file_name = os.path.basename(file_path)
        
        # Split the file name into name and extension
        name, ext = os.path.splitext(file_name)
        
        # Clean the file name: remove special characters and whitespace
        cleaned_name = re.sub(r'[^a-zA-Z0-9]', '', name)  # Keep only alphanumeric characters
        cleaned_name = cleaned_name.strip()  # Ensure no leading/trailing whitespace
        
        # Create the new file name with prefix
        new_file_name = f"{prefix}_{cleaned_name}{ext}"
        
        # Create the new file path
        new_file_path = os.path.join(directory, new_file_name)
        
        # Rename the file
        os.rename(file_path, new_file_path)
        print(f"File renamed from '{file_name}' to '{new_file_name}'")
        
        return new_file_path
    
    except PermissionError:
        print(f"Error: Permission denied while renaming '{file_path}'.")
        return None
    except FileExistsError:
        print(f"Error: A file named '{new_file_name}' already exists in the directory.")
        return None
    except Exception as e:
        print(f"Error: An unexpected issue occurred while renaming '{file_path}': {e}")
        return None
    
# Function to check if file has already been processed
def file_already_processed(collection, file_name):
    """Check if a file has already been processed by searching collection metadata."""
    try:
        # Get all metadata from the collection
        all_metadata = collection.get(
            where={"file_name": file_name}
        )
        # If there are any results with this file_name, the file was already processed
        return len(all_metadata['metadatas']) > 0
    except Exception as e:
        print(f"Error checking if file was processed: {e}")
        return False

def add_document_to_collection(file_path_new_doc, collection, storage_context, file_type="policy_country", country = "Indonesia",
                               chroma_client=chroma_client, reader=reader, node_parser=node_parser):
    """
    Add a new document to an existing collection without overwriting the existing index.
    
    Args:
        file_path_new_doc (str): Path to the new PDD document
        storage_context: chromadb storage context
        file_type (str): pdd, policy_vcm, or policy_country,
        country (str): name of the country, if file_type is policy_country
        chroma_client: ChromaDB client
        reader: Document reader
        node_parser: Node parser for transformations
    
    """    
    # Check if required parameters are provided
    if chroma_client is None or reader is None or node_parser is None:
        raise ValueError("chroma_client, reader, and node_parser must be provided")
    
    # Get the file name for metadata
    file_name = os.path.basename(file_path_new_doc)
    # split("_")[1]
    
    if file_already_processed(collection, file_name):
        print(f"File {file_name} already processed, skipping...")
        return None
    
    # Load the new document
    try:
        new_documents = reader.load_data(Path(file_path_new_doc))
    except Exception as e:
        print(f"Error loading document {file_path_new_doc}: {e}")
        return None
    
    if not new_documents:
        print(f"No content extracted from {file_name}")
        return None
    
    # Add metadata to the new documents
    for doc in new_documents:
        if doc.metadata is None:
            doc.metadata = {}
        if file_type=='pdd':
            doc.metadata.update({
                "file_name": file_name,
                "registry": file_name.split("_")[0],
                "project_code": file_name.split("_")[1].split(".")[0],
                "type": file_type
            })
        elif file_type== 'policy_vcm':
            doc.metadata.update({
                "file_name": file_name,
                "source": file_name.split("_")[0],
                "type": file_type
            })
        elif file_type=='policy_country':
            doc.metadata.update({
                "file_name": file_name,
                "country": country,
                "type": file_type
            })        
    # Create a new index with just the new documents, but using the existing storage context
    # This will add the new documents to the existing collection
    try:
        index = VectorStoreIndex.from_documents(
            documents=new_documents,
            transformations=[node_parser],
            storage_context=storage_context,
            cache=IngestionCache()
        )
        print(f"Successfully added {file_name} to the collection")
        return index
    except Exception as e:
        print(f"Error creating index for {file_name}: {e}")
        return None


## 1.1 Run: PDD Process

In [10]:
# # NOTE Change this to account for adding new pdd
# # file_name = os.path.basename(file_path_pdd)
# # construct vector store and customize storage context
# collection_pdd = chroma_client.get_or_create_collection(name="pdd_new")
# storage_context_pdd  = StorageContext.from_defaults(
#     vector_store=ChromaVectorStore(collection_pdd) 
# )

# index_pdd = add_document_to_collection(file_path_pdd, collection_pdd, storage_context_pdd, 
#                                        file_type="pdd")

# a2. LLM Service

#### 2.0 Global Setting & helper functions

In [11]:
# helper function:
import requests
import xml.etree.ElementTree as ET
from typing import Dict, Any
import google.generativeai as genai

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=GEMINI_API_KEY)

def call_llm_api(prompt: str) -> str:
    """
    Call Gemini LLM API with the provided prompt
    Args:
        prompt (str): The prompt to send to the LLM
    Returns:
        str: The LLM's response text
    """
    try:
        # Initialize the Gemini model (adjust model name as needed)
        model = genai.GenerativeModel(
            model_name="gemini-2.0-flash",  
            generation_config={
                "temperature": 0.0,        # Controls randomness (0.0 to 1.0)
                "max_output_tokens": 50000, # Max tokens in response
            }
        )

        # Generate content with the prompt
        response = model.generate_content(prompt)

        # Check if response was blocked or empty
        if not response.text:
            raise Exception("Gemini API returned no valid response")

        return response.text

    except Exception as e:
        raise Exception(f"Gemini API error: {str(e)}")

def parse_xml_response(response: str, root_tag: str) -> Dict[str, Any]:
    """
    Parse XML response from LLM into a dictionary, handling potential extra text.
    
    Args:
        response (str): Raw LLM response containing XML
        root_tag (str): Expected root tag (e.g., "project_info")
    
    Returns:
        Dict[str, Any]: Parsed XML as a dictionary
    """
    # Try to find any XML-like structure if the exact root_tag isn't found
    xml_start = response.find(f"<{root_tag}>")
    xml_end = response.rfind(f"</{root_tag}>")
    
    if xml_start == -1 or xml_end == -1:
        # Fallback: Look for any XML root tag (e.g., <information>)
        possible_start = response.find("<")
        possible_end = response.rfind(">")
        if possible_start != -1 and possible_end != -1 and possible_end > possible_start:
            xml_content = response[possible_start:possible_end + 1]
        else:
            raise Exception("Could not find XML in LLM response")
    else:
        xml_content = response[xml_start:xml_end + len(f"</{root_tag}>")]

    # Parse XML
    try:
        root = ET.fromstring(xml_content)
    except ET.ParseError as e:
        raise Exception(f"Invalid XML in LLM response: {e}")

    # Convert to dictionary recursively
    def xml_to_dict(element):
        result = {}
        # Handle attributes
        if element.attrib:
            result["@attributes"] = element.attrib
        # Handle children
        for child in element:
            child_data = xml_to_dict(child)
            if child.tag in result:
                if not isinstance(result[child.tag], list):
                    result[child.tag] = [result[child.tag]]
                result[child.tag].append(child_data)
            else:
                result[child.tag] = child_data
        # Handle text content
        text = element.text.strip() if element.text else ""
        if text and not result:
            return text
        elif text:
            result["#text"] = text
        return result

    parsed_dict = xml_to_dict(root)
    
    # If root tag doesn't match expected, warn but proceed
    if root.tag != root_tag:
        print(f"Warning: Expected root tag '{root_tag}', found '{root.tag}'. Proceeding with parsed data.")
    
    return parsed_dict


## 2.1 Function A: PDD Project Basic Info Extraction

In [25]:
def extract_doc_basicInfo(collection_name: str, project_code: str) -> Dict[str, Any]:
    """
    Extract basic project information from document using LLM with XML-formatted output
    Args:
        collection_name (str): chroma collection name of pdd 
        project_code (str): project code 
    Returns:
        Dictionary containing extracted project information
    """
    collection_name = 'pdd_new'
    pdd_collection = chroma_client.get_or_create_collection(collection_name)
    pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)
    index_pdd = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)

    retriever = index_pdd.as_retriever(similarity_top_k=20, metadata_filters={"project_code": project_code})  # Retrieve top 10 most relevant chunks
    
    query = "Please extract the following details from the provided voluntary carbon market project design document: Project's name; Brief description of the project;Location of the project (e.g., country, region);Current project status (e.g., under development, operational, completed);Project start date;Project end date;Project methodology (e.g., specific carbon offset standard or protocol used);Project size (e.g., area in hectares or total carbon credits generated)"
    retrieved_nodes = retriever.retrieve(query)
    
    # Format retrieved documents into context
    retrieved_texts = "\n\n".join([node.text for node in retrieved_nodes])
    # print(retrieved_texts)
    prompt = f"""
    Please extract the following details from the provided voluntary carbon market project design document: Project's name; Brief description of the project; Location of the project (e.g., country, region);Current project status (e.g., under development, operational, completed);Project start date;Project end date;Project methodology (e.g., specific carbon offset standard or protocol used);Project size (e.g., area in hectares or total carbon credits generated)

    ### Instructions ###
    - Extract these exact fields from the document: project name, description, location, coordinates, status, start date, end date, methodology, size.
    - Use the *exact* XML tag names as shown in the Output Format: <name>, <description>, <location>, <status>, <start_date>, <end_date>, <methodology>, <size>.
    - Output *ONLY* the XML structure—do not include any additional text, comments, `<think>` tags, markdown (```xml```), or explanations before or after the XML.
    - Wrap the output in the root tag `<project_info>`.
    - Keep <project_code> as it is 
    - If a field is missing or not found, use "Not specified" as the value. Infer project name if project name is not found.
    - Ensure the XML is well-formed and matches the Output Format exactly in structure and tag names.

    ### Output Format ###
    <project_info>
      <project_code>{project_code}</project_code>
      <name>PROJECT TITLE</name>
      <description>BRIEF DESCRIPTION</description>
      <location>LOCATION</location>
      <status>STATUS</status>
      <start_date>START DATE</start_date>
      <end_date>END DATE</end_date>
      <methodology>METHODOLOGY</methodology>
      <size>SIZE</size>
    </project_info>

    ### Document to Analyze ###
    {retrieved_texts}

    ### Final Directive ###
    Return ONLY the XML below, using the exact tag names from the Output Format, with no deviations or additional content.
    """
    
    # Call your preferred LLM API
    response = call_llm_api(prompt)
    # print(response)
    # Parse XML response
    try:
        parsed_data = parse_xml_response(response, "project_info")
        return parsed_data
    except Exception as e:
        print(f"Error parsing LLM response: {e}")
        raise Exception(f"Failed to parse LLM output: {e}")


## 2.2 Function B: PDD vs VCM_Policy Risk Analysis

----- NEW PIPELINE -----
0. Fetch forest_loss data from supabase
1. Create project summary, ask about additionality information -> project_nodes -> project_text
2. with project summary, retrieve relevant nodes from policy -> policy_nodes -> policy_text
4. based on 0,1,2,3, analyze risk -. use og prompt 

In [13]:
# CHange directory
os.chdir('/Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python')
from database import get_project_forest_loss
# get deforestation data 
# result = await get_project_forest_loss(p_code)
import re
import html

def clean_string(text: str) -> str:
    cleaned_lines = []
    for line in text.splitlines():
        # Remove all dots
        line_no_dots = line.replace('.', '')
        # Strip leading/trailing whitespace and collapse multiple spaces to one
        line_clean = re.sub(r'\s+', ' ', line_no_dots.strip())
        cleaned_lines.append(line_clean)
    # Join cleaned lines with a single newline
    return '\n'.join(cleaned_lines)

# # Example usage:
# step1 = clean_string(pdd_context_str)

def remove_glyphs(text):
    # Unescape HTML entities
    text = html.unescape(text)
    # Remove all glyph<...> tags
    text = re.sub(r'glyph<[^>]+>', '', text)
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# step2 = remove_glyphs(step1)
# print(step2)

In [14]:
async def analyze_project_risks(project_code: str, top_k: int = 10) -> List[Dict[str, Any]]:
    """
    Analyze project risks by comparing PDD against policy documents, returning XML-structured results.

    Args:
        project_code: project code 
        top_k: Number of top documents to retrieve.

    Returns:
        List of dictionaries containing risk metrics per query.
    """
        
    # Setup Chroma connections
    pdd_collection = chroma_client.get_or_create_collection("pdd_new")
    policy_collection = chroma_client.get_or_create_collection("policy_vcm")

    # Create vector stores and indexes
    from llama_index.vector_stores.chroma import ChromaVectorStore
    pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)
    policy_vector_store = ChromaVectorStore(chroma_collection=policy_collection)
    pdd_index = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)
    policy_index = VectorStoreIndex.from_vector_store(vector_store=policy_vector_store)

    filters = MetadataFilters(filters=[
        ExactMatchFilter(
            key="project_code", 
            value=project_code
        )
    ])
    pdd_retriever = pdd_index.as_retriever(similarity_top_k=top_k, filters=filters)
    policy_retriever = policy_index.as_retriever(similarity_top_k=top_k)
    # Custom prompt for XML output
    risk_template_str = (
        "You are a meticulous and critical carbon offset expert. Analyze the risk profile of a carbon offset project by comparing its project design document (PDD) "
        "with established carbon offset policy documents for the aspect: {query}.\n\n"
        "<instructions>\n"
        "- Review the PDD content: {pdd_texts}\n"
        "- Compare it against policy standards: {policy_texts}\n"
        "- Use this yearly forest loss data from Global Forest Watch, if elevant to this risk analysis. Here is the annual forest loss data of the project's location with 10km buffer: {forest_loss_data}\n"
        "- Identify potential risks in the category listed in < > in query.\n"
        "- Assign an overall risk score (0-10), impact level (Low, Medium, High), and likelihood (Unlikely, Possible, Likely).\n"
        "- Provide a brief description for the risks.\n"
        "- Provide a list of keywords, separated by comma, that describe the risks\n"
        "- Use the *exact* XML tag names as listed in after query's analyze the risk category:.\n"
        "- Output *ONLY* one <risk_category>. If there are multiple risks, then explain in the description.\n"
        "- Output *ONLY* the XML structure—do not include additional text, comments, `<think>` tags, markdown, or explanations.\n"
        "</instructions>\n\n"
        "<output_format>\n"
        "<risk_metrics>\n"
        "  <risk_category name=\"CATEGORY\">\n"
        "    <score>SCORE_VALUE</score>\n"
        "    <impact>level of impact that if this risk happens, how much will it affect the project quality</impact>\n"
        "    <likelihood>likelihood of this risk happening to this project</likelihood>\n"
        "    <description>RISK_DESCRIPTION</description>\n"
        "    <keywords>RISK_KEYWORDS</keywords>\n"
        "  </risk_category>\n"
        "</risk_metrics>\n"
        "</output_format>"
    )
    risk_template = PromptTemplate(risk_template_str)

    all_risk_metrics = []

    # reconstruct the prompt: project_query_list = [{"risk_category": "<Additionality>", "risk_description": "testing"}]
    # for question in project_query_list:
    # print(question['risk_category'])
    project_query_list = [{"risk_category": "<Additionality>", "risk_description": "analyze the risk category: <Additionality> - How does the project demonstrate that it is additional, and what evidence supports this claim? Additionality ensures that the emissions reductions or removals would not have occurred without carbon offset funding. Look for evidence such as financial barriers, technological challenges, or policy gaps that the project overcomes."},
                        {"risk_category": "<Baseline Scenario>", "risk_description": "analyze the risk category: <Baseline Scenario> - What is the baseline scenario for the project, and how was it established? The baseline scenario represents the emissions that would have occurred without the project. Check if it’s based on credible data, conservative assumptions, and an appropriate methodology for the project type."},
                        {"risk_category": "<Permanence>", "risk_description": "analyze the risk category: <Permanence> - For projects involving carbon sequestration, what measures are in place to ensure the permanence of the sequestered carbon? For projects like reforestation or soil carbon storage, permanence is critical. Ask about safeguards like buffer pools, long-term management plans, or insurance against reversals (e.g., due to fires or deforestation)."},
                        {"risk_category": "<Leakage>", "risk_description": "analyze the risk category: <Leakage> - Has the project assessed potential leakage, and how is it accounted for in the emissions reductions calculations? Leakage occurs when emissions are displaced elsewhere (e.g., deforestation shifting to another area). Verify if a leakage assessment was conducted and if mitigation measures are included."},
                        {"risk_category": "<Monitoring and Verification>", "risk_description": "analyze the risk category: <Monitoring and Verification> - What is the monitoring plan, and how will the project's emissions reductions be verified by a third party? A robust monitoring plan should detail what data will be collected, how, and how often. Confirmation of third-party verification ensures accuracy and independence."}
                        ]
    import asyncio
    # 0. fetch forest_loss data from supabase
    async def forest(project_code):
        result = await get_project_forest_loss(project_code)
        return result

    forestloss_context = await forest(project_code)

    # 1. Create project summary, ask about additionality information -> project_nodes -> project_text

    # THIS IS THE START OF THE FOR LOOP
    for q in project_query_list:
        risk_category = q['risk_category']
        question = q['risk_description']
            
        # TODO: put the following into the for loop after the test
        pdd_nodes = pdd_retriever.retrieve(question)
        pdd_context_str = "\n\n".join([node.node.get_content() for node in pdd_nodes])
        step1 = clean_string(pdd_context_str)
        pdd_context_str = remove_glyphs(step1)
        # 2. with project summary, retrieve relevant nodes from policy -> policy_nodes -> policy_text
        prompt_2_policy = f"Based on the project context {pdd_context_str} in risk catergory {risk_category}, what policy, standards, or best practices are relevant to the project?"
        policy_nodes = policy_retriever.retrieve(prompt_2_policy)
        policy_context_str = "\n\n".join([node.node.get_content() for node in policy_nodes])
        step1 = clean_string(policy_context_str)
        policy_context_str = remove_glyphs(step1)
        # 4. based on 0,1,2, analyze risk. use prompt template

        # Format the full prompt with context
        formatted_prompt = risk_template.format(
            query=question,
            project_code=project_code,
            pdd_texts=pdd_context_str,
            policy_texts=policy_context_str,
            forest_loss_data = forestloss_context
        )
        # Call the LLM directly
        response_text = call_llm_api(formatted_prompt)  # Assuming this function is defined elsewhere
        try:
            # Parse XML using the new function
            parsed_response = parse_xml_response(response_text, "risk_metrics")
            risk_metrics = []
            # Handle case where risk_category is a single dict or a list
            risk_categories = parsed_response.get("risk_category")
            if not risk_categories:
                print(f"No risk categories found for '{query}'")
                # continue
            if isinstance(risk_categories, dict):
                risk_categories = [risk_categories]
            for category in risk_categories:
                risk_metrics.append({
                    "category": category["@attributes"]["name"],
                    "score": int(category["score"]),
                    "impact": category["impact"],
                    "likelihood": category["likelihood"],
                    "description": category["description"],
                    # "query": query
                })
            all_risk_metrics.extend(risk_metrics)
            # print(f"Risk metrics for '{query}': {response_text}")
        except Exception as e:
            print(f"Error parsing risk metrics for '{query}': {e}")
            print(f"LLM response:\n{response_text}")
            continue  # Don't raise; just skip
        
    return all_risk_metrics

#### Result - PDD vs VCM

In [15]:
# #### GOGO2 LLM Analysis: - PDD vs VCM
# project_code = p_code
# all_risk_metrics = await analyze_project_risks(project_code)
# all_risk_metrics

## 2.3 Function C: PDD vs Regional Policy Risk Analysis

#### Global Function

In [16]:
import re
import json
from typing import Dict, List, Any, Optional, Union
from llama_index.core import VectorStoreIndex, PromptTemplate
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
from llama_index.core.response_synthesizers import get_response_synthesizer
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.tools import FunctionTool
# from llama_index.core.tools import QueryEngineTool
from llama_index.tools.duckduckgo import DuckDuckGoSearchToolSpec
from llama_index.core.agent.workflow import ReActAgent, FunctionAgent, AgentWorkflow
from tabnanny import verbose
# 1. A reusable decorator
import time
import functools

def rate_limit(calls_per_minute: float):
    """
    Decorator that ensures the wrapped function is called at most
    `calls_per_minute` times per minute.
    """
    interval = 60.0 / calls_per_minute
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            time.sleep(interval)
            return func(*args, **kwargs)
        return wrapper
    return decorator


In [17]:
# class JSONProcessor:
#     """
#     # Class for handling JSON processing and text cleaning operations.
#     """
    
#     @staticmethod
#     def extract_clean_json(response_str: str) -> Dict:
#         """
#         Remove markdown formatting and load JSON.
        
#         Args:
#             response_str: String containing JSON with possible markdown formatting
            
#         Returns:
#             Dict: Parsed JSON object
#         """
#         # Check if response is empty
#         if not response_str or response_str.strip() == "":
#             print("ERROR: Received empty response string")
#             return {"error": "Empty response received"}
        
#         # Print the raw response for debugging
#         print(f"Raw response: {repr(response_str[:100])}...")
        
#         # Remove markdown formatting
#         json_text = re.sub(r"^```json|```$", "", response_str.strip(), flags=re.MULTILINE)
        
#         # Try to find JSON within the text if it's not already valid JSON
#         if not json_text.strip().startswith('{'):
#             # Look for JSON-like structure
#             match = re.search(r'({[\s\S]*})', json_text)
#             if match:
#                 json_text = match.group(1)
#                 print(f"Extracted JSON-like structure: {json_text[:100]}...")
        
#         try:
#             return json.loads(json_text)
#         except json.JSONDecodeError as e:
#             print(f"JSON decode error: {e}")
#             print(f"Attempted to decode: {repr(json_text[:200])}...")
#             # Return a default structure to avoid breaking the pipeline
#             return {
#                 "project_overview": "Error parsing JSON response",
#                 "current_landuse": "Not specified",
#                 "location": ["Not specified", "Not specified"],
#                 "local_economy": "Not specified",
#                 "baseline_scenario": "Not specified",
#                 "justification_of_additionality": "Not specified",
#                 "permanence": "Not specified",
#                 "community_engagement": "Not specified",
#                 "benefit_sharing": "Not specified",
#                 "rights_and_land_tenure": "Not specified",
#                 "revenue_streams": "Not specified"
#             }

In [18]:
# Old json processor
class JSONProcessor:
    """
    Class for handling JSON processing and text cleaning operations.
    """
    
    @staticmethod
    def extract_clean_json(response_str: str) -> Dict:
        """
        Remove markdown formatting and load JSON.
        
        Args:
            response_str: String containing JSON with possible markdown formatting
            
        Returns:
            Dict: Parsed JSON object
        """
        # Remove markdown formatting
        json_text = re.sub(r"^```json|```$", "", response_str.strip(), flags=re.MULTILINE)
        return json.loads(json_text)
    
    @staticmethod
    def clean_text(text: Union[str, List, Dict, Any]) -> Union[str, List, Dict, Any]:
        """
        Clean text by removing font glyph artifacts and non-printable characters.
        Handles different types (strings, lists, dicts) appropriately.
        
        Args:
            text: Text or other data structure to clean
            
        Returns:
            Cleaned version of input, preserving the original type
        """
        # If it's a string, clean it
        if isinstance(text, str):
            # Remove font glyph artifacts
            cleaned = re.sub(r"glyph<c=\d+,font=[^>]+>", "", text)
            # Remove non-printable characters
            cleaned = re.sub(r"[^\x20-\x7E\n\r]", "", cleaned)
            return cleaned
        # If it's a list, clean each string element
        elif isinstance(text, list):
            return [JSONProcessor.clean_text(item) for item in text]
        # If it's a dictionary, clean each string value
        elif isinstance(text, dict):
            return {k: JSONProcessor.clean_text(v) for k, v in text.items()}
        # For other types (numbers, None, etc.), return as is
        else:
            return text
    
    @staticmethod
    def merge_json_outputs(json_strings: List[str]) -> Dict:
        """
        Merge multiple JSON strings into a single JSON object.
        
        Args:
            json_strings: List of JSON strings to merge
            
        Returns:
            Dict: Merged JSON object
        """
        result = {}
        
        for json_str in json_strings:
            # Remove markdown code block formatting if present
            clean_str = json_str.replace("```json", "").replace("```", "").strip()
            
            try:
                # Parse the JSON string
                data = json.loads(clean_str)
                
                # Merge with the result
                result.update(data)
            except json.JSONDecodeError as e:
                print(f"Error parsing JSON: {e}")
                print(f"Problematic JSON string: {clean_str}")
        
        return result

#### Analysis Function

In [19]:
class RegRiskAnalyzer:
    """
    Class for analyzing project risks by comparing PDDs against policy documents.
    """
    MAX_CALLS_PER_MINUTE = 5
    
    def __init__(self, project_code: str):
        """
        Initialize the RegRiskAnalyzer with required components.
        
        Args:
            project_code: project code
        """
        self.json_processor = JSONProcessor()
        self.project_code = project_code
        # self.tools = self._setup_tools()
        self._query_delay = 60.0 / self.MAX_CALLS_PER_MINUTE
            
        # ChromaDB Setup
        # Set up PDD & Policy collection and index
        pdd_collection = chroma_client.get_or_create_collection("pdd_new")
        pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)
        policy_collection = chroma_client.get_or_create_collection("regional_policy_IND")
        policy_vector_store = ChromaVectorStore(chroma_collection=policy_collection)
        
        self.pdd_index = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)
        self.policy_index = VectorStoreIndex.from_vector_store(vector_store=policy_vector_store)
        
        # Filter by project code
        self.filters = MetadataFilters(filters=[
            ExactMatchFilter(
                key="project_code", 
                value=self.project_code 
            )
        ])
        self.response_synthesizer = get_response_synthesizer()
    
    # test passed 
    def analyze_pdd_risks(self, top_k: int = 50) -> Dict[str, Any]:
        """
        Analyze project risks by comparing PDD against policy documents.
        
        Args:
            top_k (int): Number of top documents to retrieve
            
        Returns:
            Dict containing risk analysis results
        """
        # Create retriever and query engine
        pdd_retriever = self.pdd_index.as_retriever(filters=self.filters, similarity_top_k=top_k)
        pdd_query_engine = RetrieverQueryEngine(
            retriever=pdd_retriever,
            response_synthesizer=self.response_synthesizer,
        )
        
        # PDD extraction prompt
        pdd_extraction_prompt = """
            You are an AI assistant tasked with extracting specific information from a Project Design Document (PDD). Your goal is to output a JSON object with the following fixed keys, ensuring that each key is present in the output. If a particular piece of information is not available in the PDD, use "Not specified" as the value.

            Please adhere strictly to the following JSON structure:

            {
            "project_overview": "Provide a concise summary of the project's objectives and key design elements as a carbon offset initiative.",
            "current_landuse": "Describe the designated land use of the project area prior to implementation.",
            "location": "Specify the project's location, including the province or region. return a list [name of location, province]",
            "local_economy": "Summarize the characteristics of the local economy in the project area.",
            "baseline_scenario": "Detail the baseline scenario, outlining what would occur in the absence of the project, and how this baseline is created using what methodologies",
            "justification_of_additionality": "How does the project demonstrate that the carbon benefits would not occur without its implementation?",
            "permanence": "How does the project ensure long-term carbon sequestration, and what measures are in place to address potential reversals (e.g., wildfires, logging)?"
            "community_engagement": "How were local communities consulted during project development, and do they have mechanisms for ongoing participation?  Explain."
            "benefit_sharing": "Does the project provide tangible benefits to local stakeholders, such as employment or revenue sharing?  Explain."
            "rights_and_land_tenure": "Are land rights and tenure issues clearly addressed, ensuring that the project does not infringe upon indigenous or local communities' rights? Explain."
            "revenue_streams": "What are the projected revenue sources, and are they diversified to ensure financial stability?",        
            }

            Ensure that:
            - All keys are included in the output JSON.
            - The values are extracted directly from the PDD content.
            - The output is a valid JSON object without any explanatory text or commentary.
        """

        # Query PDD and process result
        pdd_basicinfo = pdd_query_engine.query(pdd_extraction_prompt)
        print(pdd_basicinfo)
        clean_data = self.json_processor.extract_clean_json(pdd_basicinfo.response)
        # print(f"clean data is {clean_data}")
        cleaned_json = {k: self.json_processor.clean_text(v) for k, v in clean_data.items()}
        
        return cleaned_json
    
    # test passed
    def analyze_policy_risks(self, project_data_riskAnalysis: Dict[str, Any], 
                            top_k: int = 10) -> Dict[str, Any]:
        """
        Analyze policy risks based on the project data.
        
        Args:
            project_data_riskAnalysis: Cleaned project data from analyze_regional_risks
            top_k: Number of top documents to retrieve
            
        Returns:
            Dict containing policy risk analysis
        """
        
        # Create retriever and query engine
        policy_retriever = self.policy_index.as_retriever(similarity_top_k=top_k)
        policy_query_engine = RetrieverQueryEngine(
            retriever=policy_retriever,
            response_synthesizer=self.response_synthesizer,
        )
        
        # Risk template
        risk_template_str = (
            """Analyze the risk profile of a carbon offset project by comparing its project design document (PDD)
            with established national and regional forestry policy documents for the aspect: {query}.
            Here are some context of the {pdd_risk_texts} from the PDD. If there are abbreviations, list the full names in a bracket.
            
            IMPORTANT: Your response MUST be a valid JSON object with exactly ONE key (the category name) and ONE string value (your analysis).
            For example: {"risk category": "Your comprehensive analysis here. Can contain one level of nests json to break down topics under risk category analysis"}
    
            Ensure that:
            - The output is a valid JSON object with a single key-value pair
            - The key matches the category name in the query
            - The value is a comprehensive string containing your full analysis
            """
        )
        risk_template = PromptTemplate(risk_template_str)
        
        # List of project queries
        project_query_list = [
            "risk category: <regulatory> - Are forest conservation, mangrove restoration, or ecosystem services programs already supported in [location/province] through national or regional plans? Does Indonesian forestry policy already include similar forest protection efforts in this area? What are the official land use classifications and restrictions for [location/province]? Is the project area already protected under Indonesian forestry or conservation law / Is [project location] part of a moratorium, protected forest, or conservation area under national/regional law? Are there reforestation, REDD+, or forest protection programs already underway here?",
            "risk category: <finance> - Do government grants, public incentives, or subsidies already exist for forest protection or restoration in this area? Does policy or funding from MoEF or local government already enable these project activities regardless of carbon financing?",
            "risk category: <permanence> - Does the government have forest patrols, peatland restoration, or wildfire prevention programs in this area? What regulations are in place to safeguard protected forests in [location]?",
            "risk category: <local_economy> - Are there national or local programs addressing economic issues(e.g. illegal logging, unsustainable agriculture) in [location]? Are there alternative livelihood programs supported by the government in this region?",
        ]
        
        # Process each query
        all_risk_metrics = []
        for query in project_query_list:
            # Format the full prompt with context
            formatted_prompt = risk_template.format(
                query=query,
                pdd_risk_texts=project_data_riskAnalysis,
            )
            # Call the query engine
            response_text = policy_query_engine.query(formatted_prompt)
            all_risk_metrics.append(response_text.response)
    
        # Merge and return results
        return self.json_processor.merge_json_outputs(all_risk_metrics)
    
    # creating functions and tools for agents to use
    @rate_limit(MAX_CALLS_PER_MINUTE)
    def _pdd_query_engine(self, query: str) -> str:
        """
        Answer questions regarding project design document (PDD).
        
        Args:
            query: The question to ask about the PDD
            
        Returns:
            str: Response to the query
        """
        pdd_retriever = self.pdd_index.as_retriever(filters=self.filters, 
                                                    similarity_top_k=5)
                
        # Assemble query engine
        pdd_query_engine = RetrieverQueryEngine(
            retriever=pdd_retriever,
            response_synthesizer=self.response_synthesizer,
        )
        return pdd_query_engine.query(query).response
    
    @rate_limit(MAX_CALLS_PER_MINUTE)
    def _policy_query_engine(self, query: str) -> str:
        """
        Answer questions regarding national and regional forest and land policy.
        
        Args:
            query: The question to ask about policy
            
        Returns:
            str: Response to the query
        """
        policy_retriever = self.policy_index.as_retriever(similarity_top_k=5)
        
        # Assemble query engine
        policy_query_engine = RetrieverQueryEngine(
            retriever=policy_retriever,
            response_synthesizer=self.response_synthesizer,
        )
        return policy_query_engine.query(query).response

 # Running the first agent 
    async def agent_workflow(self, prompt):
        """
        Summary:
            3 agents: pi(ReAct Agent), gapfiller(Function Agent), socialRA(Function Agent)
        """
        #  tools:
        # 1. New Web Search Tool
        from llama_index.tools.tavily_research.base import TavilyToolSpec
        tavily_tool = TavilyToolSpec(
            api_key=API_TAV,
        )
        web_search_tool = FunctionTool.from_defaults(
            tavily_tool.search,
            name="web_search",
        )
        
        # 2. PDD query tool
        pdd_query_tool = FunctionTool.from_defaults(
            self._pdd_query_engine,
            name="pdd_query_engine_tool",
            description="A tool to answer questions regarding project design document (PDD)",
        )
        
        # 3. Policy query tool
        policy_query_tool = FunctionTool.from_defaults(
            self._policy_query_engine,
            name="policy_query_engine_tool",
            description="A tool to answer questions regarding national and regional forest and land policy",
        )
        # # agent 1: PI - review and synthesize result, ensure correct format
        # agent_pi = ReActAgent(
        #     name="PIAgent",
        #     description="Review content for accuracy and quality",
        #     system_prompt=(
        #         """You are a meticulous reviewer to make sure langauges are correct and that the final output is in correct json format.
        #         Create a comprehensive analysis complete analysis by getting the information from GapFillerAgent regarding policy and SocialAgent regarding social equity. Review, correct the language, and compile.
        #         STOP once you have a comprehensive answer - do not make unnecessary follow-up queries
        #         Be efficient with your tool usage and aim to provide complete answers in as few steps as possible.
                
        #         Output a json object with the following structure. Each key can contain one level of nests json to break down topics under search analysis:
        #         {
        #             "regulatory": "Your comprehensive analysis here",
        #             "finance": "Your comprehensive analysis here. ",
        #             "permanence": "Your comprehensive analysis here.",
        #             "local_economy": "Your comprehensive analysis here.",
        #             "social_equity": "Your comprehensive analysis here."
        #         }
        #         """
        #         ),
        #     can_handoff_to=["GapFillerAgent", "SocialAgent"]
        # )
        
        # # agent 2: gap filler - fill gaps in the policy analysis
        # agent_gapfiller = FunctionAgent(
        #     name="GapFillerAgent",
        #     #llm = llm_hf,
        #     description="Searches and analyzes information from multiple sources",
        #     tools = self.tools, # policy_query_tool, pdd_query_tool, web_search_tool
        #     # TODO: this needs to be fixed to reflect gap filler, 
        #     # given context of the pdd and policy analysis, fill any gaps, 
        #     # augment the analysis with web search as needed, but not too much web search
        #     system_prompt=(
        #         """You are an expert researcher in the voluntary carbon offset market and policy analysis, focusing on forestry and landuse project.
                
        #         Your task is to review the pdd_info and policy_info analysis and try to improve the content:
        #         1. Review the information from PDD and policy from prompt. Reason through and analyze the retrieved information and identify gaps that need more context for analysis. 
        #         2. If there are information lacking from the policy, call the policy_query_engine_tool; call pdd_query_tool to get more information from pdd; call web_search_tool (Limit the number of search in the web search tool to 3) for additional information.
        #         3. Compile a concise, complete answer
        #         4. STOP once you have a comprehensive answer - do not make unnecessary follow-up queries
                
        #         Output a json object with the following structure. Each key can contain one level of nests json to break down topics under search analysis:
        #         {
        #             "regulatory": "Your comprehensive analysis here",
        #             "finance": "Your comprehensive analysis here. ",
        #             "permanence": "Your comprehensive analysis here."
        #             "local_economy": "Your comprehensive analysis here."
        #             "social_equity" : "Your analysis"
        #         }
                
        #         Be efficient with your tool usage and aim to provide complete answers in as few steps as possible.
        #         """
        #     ),
        #     can_handoff_to=["SocialAgent"]
        # )
        
        # agent 3: social analysis researcher 
        agent_social = FunctionAgent(
            name="SocialAgent",
            description="Searches and analyzes information from multiple sources regarding social and equity issues",
            tools = [web_search_tool], # policy_query_tool, pdd_query_tool, search_tool        
            system_prompt=(
                """
                You are a social science researcher focusing on carbon offset market social and equity issue.
                Look for news, reports, or articles regarding social/equity issues in the location.
                Output json format: {"title": "", "url": "", "content": ""},.
                """
            ),
            # can_handoff_to=["GapFillerAgent"]
        )
        
        # Create the workflow
        workflow = AgentWorkflow(
            agents=[agent_social],
            root_agent="SocialAgent",
            verbose=True
        )
        # TODO: Debug this -> asyncio issue 
        result = await workflow.run(prompt)
        # NOTE This may or may notwork lol
        self.json_processor.extract_clean_json(result.response.blocks[0].text)
        return result 

# # change the model to 1.5 flash to try to get more results ...
# api_key = os.getenv("GEMINI_API_KEY")
# llm_agent = Gemini(model_name="models/gemini-2.0-flash", 
#                 temperature=0.0, 
#                 max_tokens=100000, 
#                 api_key=api_key)
# Settings.llm = llm_agent
# Results from the analysis

# !!!GOGO1-CreateProject-Index and Project Basic Info

In [59]:
# NOTE: Change things here
p_code = '1899'
# NOTE: Change things here

# PDD FILE MUST START WITH REGISTRY CODE & PROJECT CODE!!!!
os.chdir('/Users/beckyxu/Documents/GitHub/Carbon_Offset_Validation/carbon-offset-validator/server-python')
file_path_pdd = f'_pipeline/data/pdd/VCS_{p_code}.pdf'
print(file_path_pdd)

_pipeline/data/pdd/VCS_1899.pdf


In [60]:
### 1. create pdd index and add to chromadb
collection_pdd = chroma_client.get_or_create_collection(name="pdd_new")
storage_context_pdd  = StorageContext.from_defaults(
    vector_store=ChromaVectorStore(collection_pdd) 
)
index_pdd = add_document_to_collection(file_path_pdd, collection_pdd, storage_context_pdd, 
                                       file_type="pdd")

Successfully added VCS_1899.pdf to the collection


In [61]:
# 2. Create project basic info and store result to Supabase
project_data = extract_doc_basicInfo(collection_name="pdd_new", project_code=p_code)
project_data

{'project_code': '1899',
 'name': 'Katingan Mentaya Project',
 'description': 'Reducing Emissions from Deforestation and Degradation (REDD) through Avoided Planned Deforestation (APD). The project is a single, standalone project, not a grouped project.',
 'location': 'Kalimantan, Indonesia',
 'status': 'Operational',
 'start_date': 'July, 2009',
 'end_date': 'Not specified',
 'methodology': 'VCS Version 3, CCB Standards Third Edition',
 'size': '149,800 ha'}

In [62]:
# 3. Upload the project info
from database import supabase, store_analysis_results
project_id = await store_analysis_results(project_code=p_code, project_data=project_data)

Update response status: N/A
Updated Project basic info table for project with code: 1899


# !!!GOGO2 - LLM Analysis * 3

1. PDD vs VCM
2. PDD vs National
1. PDD Overall

In [63]:
from database import update_vcm_analy_summary, update_regional_analy_summary, upload_overall_analysis

In [64]:
#### LLM Analysis 1: PDD vs VCM
# NOTE! Wait for Forest Loss Data Here!
project_code = p_code
all_risk_metrics = await analyze_project_risks(project_code)
all_risk_metrics

[{'category': 'Additionality',
  'score': 6,
  'impact': 'Medium',
  'likelihood': 'Possible',
  'description': "The PDD states that the most credible baseline is the development of an industrial paper pulp plantation, citing a GGGI study that supports this as the most profitable land use. However, the PDD also mentions that a detailed investment analysis is not required by the A/R CDM additionality tool because a single credible land-use scenario was identified. This lack of a detailed investment analysis, combined with the high deforestation rates in the region (as indicated by the Global Forest Watch data), raises concerns about whether the project is truly additional. While the project includes activities like Avoiding Planned Deforestation (APD), Reforestation (ARR), Conservation of Undrained and Partially-drained Peatland (CUPP) and Rewetting of Drained Peatland (RDP), the financial attractiveness of industrial acacia plantations could still pose a risk to the project's additiona

In [65]:
# Upload PDD vs VCM analysis
await update_vcm_analy_summary(p_code, all_risk_metrics)

Querying Supabase for project with code: 1899
Found project with ID: 8183bf76-bf9c-43b2-aed9-8f58b5d5c56b
Checking for existing summary for project_id: 8183bf76-bf9c-43b2-aed9-8f58b5d5c56b
Inserting new metric for category 'Additionality'
Inserting new metric for category 'Baseline Scenario'
Inserting new metric for category 'Permanence'
Inserting new metric for category 'Leakage'
Inserting new metric for category 'Monitoring and Verification'
Risk summary metrics updated/inserted for project with code: 1899


True

In [ ]:
### LLM Analysis 2: PDD vs National (1/3)
# Results from the analysis
# add a timer to sleep for 1 min
import time
# time.sleep(120) 
## Sleep for 120 seconds (2 minute)

analyzer = RegRiskAnalyzer(p_code)
pdd_analysis = analyzer.analyze_pdd_risks(top_k=50)
print(pdd_analysis)

```json
{
"project_overview": "The project credibly avoids peatland emissions through protection and rehabilitation, restores forest types to their original state, restores hydrological integrity, and prevents fires through training, monitoring, and prevention procedures.",
"current_landuse": "Legally designated for production activities.",
"location": ["Sumatra", "Indonesia"],
"local_economy": "Not specified",
"baseline_scenario": "The most credible baseline is the development of an industrial paper pulp plantation, as there are no financial, technological, institutional, or capacity barriers. The baseline is created using industrial Acacia plantation.",
"justification_of_additionality": "Benefits to the climate, communities and biodiversity would be negatively impacted in the absence of the project.",
"permanence": "Not specified",
"community_engagement": "Stakeholder consultation, livelihood development, and education program.",
"benefit_sharing": "The project provides education opp

In [67]:
# To upload 1 - policy analysis
time.sleep(120)  # Sleep for 120 seconds (1 minute)
policy_analysis = analyzer.analyze_policy_risks(pdd_analysis)
print(policy_analysis)

{'regulatory': "To assess the regulatory risks for a carbon offset project in Sumatra, Indonesia, several aspects of Indonesian forestry policy need to be examined in relation to the project's location and activities. The project aims to avoid peatland emissions through protection and rehabilitation, restore forest types, restore hydrological integrity, and prevent fires. The project area is legally designated for production activities. A key consideration is whether forest conservation, mangrove restoration, or ecosystem services programs are already supported in Sumatra through national or regional plans. Indonesian forestry policy may already include similar forest protection efforts in this area. Official land use classifications and restrictions for Sumatra need to be checked to ensure the project aligns with these regulations. It is crucial to determine if the project area is already protected under Indonesian forestry or conservation law, or if the project location is part of a 

In [68]:
# find the location of the project
print(pdd_analysis['location'])

['Sumatra', 'Indonesia']


In [69]:
# PDD vs National (2/3)
# To upload 2 - news 
prompt_news = f"""
Find and analyze social economic news that may be related to forest or carbon market in 
{", ".join(pdd_analysis['location'])},
"""
news = await analyzer.agent_workflow(prompt_news)
newsresult = news.response.blocks[0].text
newcleaned = JSONProcessor().extract_clean_json(newsresult)
newcleaned

Running step init_run
Step init_run produced event AgentInput
Running step setup_agent
Step setup_agent produced event AgentSetup
Running step run_agent_step
Step run_agent_step produced event AgentOutput
Running step parse_agent_output
Step parse_agent_output produced no event
Running step call_tool
Step call_tool produced event ToolCallResult
Running step aggregate_tool_results
Step aggregate_tool_results produced event AgentInput
Running step setup_agent
Step setup_agent produced event AgentSetup
Running step run_agent_step
Step run_agent_step produced event AgentOutput
Running step parse_agent_output
Step parse_agent_output produced event StopEvent


[{'title': 'Low-carbon transitions in West Sumatra, Indonesia: gender and equity dimensions',
  'url': 'https://www.sei.org/publications/low-carbon-transitions-in-west-sumatra-indonesia-gender-and-equity-dimensions/',
  'content': 'This SEI policy paper explores how gender equality and social equity issues evolve in an energy-transition reality. As Indonesia begins its process towards an uneasy renewable-energy transition, it is important to prioritize gender and social equity considerations in more holistic energy planning. In decision-making on renewable-energy policy, a gender-sensitive approach could seek to understand how policies will affect members of different social groups and to find solutions to achieve equitable outcomes.'},
 {'title': 'Low-carbon transitions in West Sumatra, Indonesia',
  'url': 'https://www.sei.org/wp-content/uploads/2020/07/low-carbon-transitions-in-west-sumatra-indonesia.pdf',
  'content': 'The aim of this brief is to draw lessons using real-world examp

step 3: agents to fill in missing information
tools: web search, pdd query engine, policy query engine
agent1(Function)-gapfiller-
agent2(Function)-newsearch
agent3(ReAct)-writing&formating-


In [70]:
# PDD vs National (3/3)
# To upload 3 - summary of the analysis
time.sleep(10)  # Sleep for 60 seconds (1 minute)
prompt_regional_summary = f""" Based on the voluntary carbon offset project analysis {pdd_analysis} (project claim), 
the national/regional policy analysis {policy_analysis} (policy discrepancy),
and the web search article regarding relevant regional news {news},
write a concise summary (max.5 sentences) of analysis result highlight the discrepancy between the project claim and policy discrepancy.
"""
summary_regional = call_llm_api(prompt_regional_summary)
summary_regional

"The carbon offset project in Sumatra aims to avoid peatland emissions and restore forests, but faces regulatory risks due to its location being legally designated for production activities, potentially conflicting with existing land use designations. While the project claims community engagement and livelihood development, the policy analysis reveals a lack of clarity on government support for such initiatives and alternative livelihood programs to combat illegal logging and unsustainable agriculture. Land tenure issues and deforestation pressures, as highlighted in the web search, further complicate the project's viability. The project's additionality is questionable, as it needs to demonstrate that its activities are additional to what would have occurred under existing regulations and policies. Overall, the project's claims of positive community impact and environmental benefits are challenged by regulatory uncertainties and potential conflicts with existing land use policies.\n"

In [71]:
### LLM Analysis 2: PDD vs National (1/3)
# upload 
# NOTE this shit may note work
await update_regional_analy_summary(p_code, summary_regional, policy_analysis, newcleaned)

Querying Supabase for project with code: 1899
Found project with ID: 8183bf76-bf9c-43b2-aed9-8f58b5d5c56b
Checking for existing summary for project_id: 8183bf76-bf9c-43b2-aed9-8f58b5d5c56b
Summary data prepared: {'project_id': '8183bf76-bf9c-43b2-aed9-8f58b5d5c56b', 'summary': "The carbon offset project in Sumatra aims to avoid peatland emissions and restore forests, but faces regulatory risks due to its location being legally designated for production activities, potentially conflicting with existing land use designations. While the project claims community engagement and livelihood development, the policy analysis reveals a lack of clarity on government support for such initiatives and alternative livelihood programs to combat illegal logging and unsustainable agriculture. Land tenure issues and deforestation pressures, as highlighted in the web search, further complicate the project's viability. The project's additionality is questionable, as it needs to demonstrate that its activit

True

In [72]:
### LLM Analysis 2: PDD all - LLM analysis: create a COMBINED overall summary

prompt_all_summmary = f"""
You are a principal investigator specialized in forestry carbon offset projects.

Based on:
- The voluntary carbon offset project analysis (project claim)Based on the voluntary carbon offset project analysis {pdd_analysis} (project claim), 
- The national/regional policy analysis {policy_analysis} (policy discrepancy),
- The web search article regarding relevant regional news {news},
- And the analysis of risk that compares PDD with standards from VCM registries  {"\n\n".join([cat["description"] for cat in all_risk_metrics])},

Please do the following:
1. Write a concise abstract of the overall analysis result, highlighting the discrepancies between the project claim, policy discrepancy, and VCM standards. This should be labeled as 'Summary'.
2. Provide a few actionable recommendations in bullet points, labeled as 'Recommendations'.

Format your response as follows:

Summary:
<Your concise abstract here>

Recommendations:
- <Recommendation 1>
- <Recommendation 2>
- ...
"""
summary_all = call_llm_api(prompt_all_summmary)
summary_all

"Summary:\nThis carbon offset project in Sumatra, Indonesia, aiming to avoid peatland emissions and restore forests, faces significant challenges regarding additionality, permanence, and leakage. The project claims a baseline of industrial Acacia plantations, but lacks a detailed investment analysis to support this assumption, raising concerns about whether the project is truly additional. High deforestation rates in the region and potential for activity shifting further threaten additionality. While the project addresses permanence risks with a buffer, external pressures like fire and illegal logging remain significant. The assumption of zero leakage, despite regional deforestation, is a major discrepancy. The monitoring plan, while comprehensive, lacks specific details and relies on assumptions that could lead to overestimation of emission reductions. Indonesian forestry policies, while supportive of conservation, may overlap with project activities, requiring careful alignment. Over

In [73]:
# Upload overall Summary
from database import upload_overall_analysis
await upload_overall_analysis(project_id, summary_all)

Uploaded overall analysis for project_id 8183bf76-bf9c-43b2-aed9-8f58b5d5c56b


APIResponse[TypeVar](data=[{'id': 'c49c700c-67b3-405e-8c22-37a85b033f2e', 'project_id': '8183bf76-bf9c-43b2-aed9-8f58b5d5c56b', 'summary': "This carbon offset project in Sumatra, Indonesia, aiming to avoid peatland emissions and restore forests, faces significant challenges regarding additionality, permanence, and leakage. The project claims a baseline of industrial Acacia plantations, but lacks a detailed investment analysis to support this assumption, raising concerns about whether the project is truly additional. High deforestation rates in the region and potential for activity shifting further threaten additionality. While the project addresses permanence risks with a buffer, external pressures like fire and illegal logging remain significant. The assumption of zero leakage, despite regional deforestation, is a major discrepancy. The monitoring plan, while comprehensive, lacks specific details and relies on assumptions that could lead to overestimation of emission reductions. Indon

------

# FIX BUGSSSS EWWWWW

In [74]:
# # Update project_code in ChromaDB for documents with file_name VCS_5458.pdf
# def update_project_code_in_chroma(collection_name, old_file_name, new_project_code):
#     collection = chroma_client.get_collection(collection_name)
    
#     # Get all documents with the specified file_name
#     results = collection.get(
#         where={"file_name": old_file_name}
#     )
    
#     if not results or not results['ids']:
#         print(f"No documents found with file_name: {old_file_name} in collection {collection_name}")
#         return
    
#     print(f"Found {len(results['ids'])} documents to update in collection {collection_name}")
    
#     # For each document, update the metadata
#     for i, doc_id in enumerate(results['ids']):
#         # Get the current metadata
#         current_metadata = results['metadatas'][i]
        
#         # Update the project_code
#         current_metadata['project_code'] = new_project_code
        
#         # Update the document in the collection
#         collection.update(
#             ids=[doc_id],
#             metadatas=[current_metadata]
#         )
    
#     print(f"Updated {len(results['ids'])} documents in collection {collection_name}")

# # Update in all collections
# collections_to_update = ["pdd_new"]  # Add other collection names if needed
# file_name = "VCS_5458.pdf"
# new_project_code = '5458'

# for collection_name in collections_to_update:
#     try:
#         update_project_code_in_chroma(collection_name, file_name, new_project_code)
#     except Exception as e:
#         print(f"Error updating collection {collection_name}: {e}")

In [75]:
# TEST CHROMADB INDEX

# pdd_collection = chroma_client.get_or_create_collection("pdd_new")
# pdd_vector_store = ChromaVectorStore(chroma_collection=pdd_collection)

# pdd_index = VectorStoreIndex.from_vector_store(vector_store=pdd_vector_store)

# # Filter by project code
# filters = MetadataFilters(filters=[
#     ExactMatchFilter(
#         key="project_code", 
#         value='5458' 
#     )
# ])
# response_synthesizer = get_response_synthesizer()

# pdd_retriever = pdd_index.as_retriever(filters=filters, similarity_top_k=2)
# pdd_query_engine = RetrieverQueryEngine(
#     retriever=pdd_retriever,
#     response_synthesizer=response_synthesizer,
# )
# pdd_query_engine.query("what is the main goal of this project")